In [1]:
"""
Train and save all models, scalers, SHAP explainers, and metrics.
Run this once before launching the dashboard.
"""

import pandas as pd
import numpy as np
import joblib
import warnings
import os
import json
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import xgboost as xgb
import shap

os.makedirs("models", exist_ok=True)

# ── Data ──────────────────────────────────────────────────────────────────────
print("Loading data...")
df = pd.read_csv('Heart.csv', na_values=['NA', '?', ''])
df['AHD'] = df['AHD'].map({'No': 0, 'Yes': 1})
X = df.drop(columns=['AHD', 'HD'])
y = df['AHD']

num_features = ['Age', 'RestBP', 'Chol', 'MaxHR', 'Oldpeak', 'Ca']
cat_features = ['Sex', 'ChestPain', 'Fbs', 'RestECG', 'ExAng', 'Slope', 'Thal']
feature_names = num_features + cat_features

print("  Loaded Heart Disease dataset.")

# ── Preprocessing ─────────────────────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
], remainder='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)

joblib.dump((X_test, y_test), "models/test_data.pkl")
joblib.dump(feature_names, "models/feature_names.pkl")
joblib.dump(preprocessor, "models/preprocessor.pkl")
print("  Preprocessing done.")

# Calculate positive class weight for imbalanced datasets
pos_weight_train = (y_train == 0).sum() / (y_train == 1).sum()

# ── Model definitions ─────────────────────────────────────────────────────────
model_configs = {
    "Random Forest": {
        "pipeline": Pipeline([
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'))
        ]),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [None, 10, 15],
            "model__min_samples_split": [2, 5],
            "model__class_weight": ['balanced', 'balanced_subsample']
        },
    },
    "XGBoost": {
        "pipeline": Pipeline([
            ("preprocessor", preprocessor),
            ("model", xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1, scale_pos_weight=pos_weight_train))
        ]),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [3, 4, 5],
            "model__learning_rate": [0.05, 0.1, 0.2],
            "model__subsample": [0.8, 1.0]
        },
    },
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(max_iter=500, random_state=42))
        ]),
        "params": {"model__C": [0.01, 0.1, 1, 10, 100], "model__penalty": ["l2"]},
    },
    "KNN": {
        "pipeline": Pipeline([
            ("preprocessor", preprocessor),
            ("model", KNeighborsClassifier())
        ]),
        "params": {"model__n_neighbors": [3,5,7,9,11,13],
                   "model__weights": ["uniform","distance"]},
    },
    "SVM": {
        "pipeline": Pipeline([
            ("preprocessor", preprocessor),
            ("model", SVC(probability=True, random_state=42))
        ]),
        "params": {"model__kernel": ["linear","rbf"],
                   "model__C": [0.1,1,10,100],
                   "model__gamma": ["scale",0.1,0.01]},
    },
}

metrics_all = {}
roc_data = {}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, cfg in model_configs.items():
    print(f"\nTraining {name}...")
    grid = GridSearchCV(cfg["pipeline"], cfg["params"], cv=cv, scoring="roc_auc", n_jobs=-1)
    grid.fit(X_train, y_train)
    best = grid.best_estimator_
    print(f"  Best params: {grid.best_params_}")

    safe_name = name.lower().replace(" ", "_")
    joblib.dump(best, f"models/{safe_name}.pkl")

    pred = best.predict(X_test)
    prob = best.predict_proba(X_test)[:, 1]

    metrics_all[name] = {
        "Accuracy":  round(accuracy_score(y_test, pred), 4),
        "Precision": round(precision_score(y_test, pred), 4),
        "Recall":    round(recall_score(y_test, pred), 4),
        "F1 Score":  round(f1_score(y_test, pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, prob), 4),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist()
    }

    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_data[name] = {"fpr": fpr.tolist(), "tpr": tpr.tolist(),
                       "auc": metrics_all[name]["ROC-AUC"]}

    # ── SHAP ──────────────────────────────────────────────────────────────────
    print(f"  Computing SHAP for {name}...")
    
    # Transform data once so we don't pass raw DataFrames through the pipeline repeatedly
    X_train_trans = best.named_steps['preprocessor'].transform(X_train)
    X_test_trans = best.named_steps['preprocessor'].transform(X_test[:80])
    
    # For tree-based models, use TreeExplainer; for others, use KernelExplainer
    if name in ["Random Forest", "XGBoost"]:
        # Extract the trained model from the pipeline
        trained_model = best.named_steps['model']
        explainer = shap.TreeExplainer(trained_model)
        
        shap_vals = explainer.shap_values(X_test_trans)
        
        if isinstance(shap_vals, list):
            sv_positive = shap_vals[1] if len(shap_vals) > 1 else shap_vals[0]
        else:
            sv_positive = shap_vals
            
        ev = explainer.expected_value
        if hasattr(ev, '__len__') and len(ev) > 1:
            ev = float(ev[1])
        else:
            ev = float(ev)
            
    else:
        # For non-tree models, use KernelExplainer
        background_arr = shap.sample(X_train_trans, 80, random_state=42)
        
        # FIX: Extract the actual estimator from the pipeline and pass its predict_proba method.
        # Passing the whole pipeline causes it to try and re-transform the already-transformed NumPy array.
        trained_model = best.named_steps['model']
        explainer = shap.KernelExplainer(trained_model.predict_proba, background_arr)
        shap_vals = explainer.shap_values(X_test_trans, nsamples=100)
        
        if isinstance(shap_vals, list):
            sv_positive = shap_vals[1]
        else:
            sv_positive = shap_vals
            
        ev = explainer.expected_value
        if hasattr(ev, '__len__'):
            ev = float(ev[1])
        else:
            ev = float(ev)
    
    # Calculate feature importance
    importance = np.abs(sv_positive).mean(axis=0)
    fi_df = pd.DataFrame({"Feature": feature_names, "Mean_SHAP": importance.tolist()})
    fi_df = fi_df.sort_values("Mean_SHAP", ascending=False)

    joblib.dump({
        "expected_value": ev,
        "shap_values_positive": sv_positive,
        "X_test_sample": pd.DataFrame(X_test_trans, columns=feature_names) if name not in ["Random Forest", "XGBoost"] else X_test[:80].reset_index(drop=True),
        "feature_importance": fi_df,
        "is_tree_based": name in ["Random Forest", "XGBoost"]
    }, f"models/{safe_name}_shap.pkl")

    print(f"  SHAP saved for {name}.")

with open("models/metrics.json", "w") as f:
    json.dump(metrics_all, f, indent=2)
with open("models/roc_data.json", "w") as f:
    json.dump(roc_data, f, indent=2)

print("\n✅ All models trained and saved to ./models/")

Loading data...
  Loaded Heart Disease dataset.
  Preprocessing done.

Training Random Forest...
  Best params: {'model__class_weight': 'balanced_subsample', 'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 200}
  Computing SHAP for Random Forest...
  SHAP saved for Random Forest.

Training XGBoost...
  Best params: {'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__n_estimators': 100, 'model__subsample': 0.8}
  Computing SHAP for XGBoost...
  SHAP saved for XGBoost.

Training Logistic Regression...
  Best params: {'model__C': 1, 'model__penalty': 'l2'}
  Computing SHAP for Logistic Regression...


  0%|          | 0/61 [00:00<?, ?it/s]

  SHAP saved for Logistic Regression.

Training KNN...
  Best params: {'model__n_neighbors': 13, 'model__weights': 'distance'}
  Computing SHAP for KNN...


  0%|          | 0/61 [00:00<?, ?it/s]

  SHAP saved for KNN.

Training SVM...
  Best params: {'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
  Computing SHAP for SVM...


  0%|          | 0/61 [00:00<?, ?it/s]

  SHAP saved for SVM.

✅ All models trained and saved to ./models/
